In [1]:
import hail as hl
import logging
import pickle
from itertools import combinations
from typing import Dict, List, Set, Tuple, Union
import numpy as np
import pandas as pd
from gnomad.resources.resource_utils import DataException
from gnomad.utils.file_utils import file_exists

In [2]:
hl.init(
    spark_conf={
        'spark.hadoop.fs.gs.requester.pays.mode': 'CUSTOM',
        'spark.hadoop.fs.gs.requester.pays.buckets': 'regional_missense_constraint,gnomad-public-requester-pays,gnomad',
        'spark.hadoop.fs.gs.requester.pays.project.id': 'lily-sandbox-a29d'
    }, default_reference='GRCh38'
)

/opt/conda/miniconda3/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning: Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.
  warnings.warn(
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 56323
SPARKMONITOR_LISTENER: Application Started: application_1770746110311_0002 ...Start Time: 1770752793746


Running on Apache Spark version 3.3.0
SparkUI available at http://lw-m.us-central1-b.c.lily-sandbox-a29d.internal:44611
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.120-f00f916faf78
LOGGING: writing to /home/hail/hail-20260210-1946-0.2.120-f00f916faf78.log


The purpose of this notebook is create an environment to be able to experiment with MPC using different training model architectures and evaluate relevant metrics to assess performance.

All necessary resource paths are imported from main RMC repository. 

In [3]:
from rmc.resources.basics import (    
    MPC_PREFIX,
    TEMP_PATH_WITH_FAST_DEL,
    TEMP_PATH_WITH_SLOW_DEL,
)
from rmc.resources.gnomad import constraint_ht
from rmc.resources.reference_data import (
    FOLD_K,
    get_ref_data_prefix,
    blosum,
    blosum_txt_path,
    cadd,
    grantham,
    grantham_txt_path,
    train_val_test_transcripts_path,
)
from rmc.resources.resource_utils import CURRENT_GNOMAD_VERSION
from rmc.resources.rmc import (
    CURRENT_FREEZE,
    context_with_oe,
    context_with_oe_dedup,
    filtered_context,
    gnomad_fitted_score_path,
    joint_clinvar_gnomad_path,
    misbad_path,
    mpc_model_pkl_path,
    mpc_release,
)
from rmc.utils.constraint import get_constraint_transcripts, explode_intervals_to_loci
from rmc.utils.generic import (
    get_aa_map,
    get_gnomad_public_release,
    keep_criteria,
)
from rmc.utils.missense_badness import variant_csq_expr
from rmc.utils.mpc import (
    convert_score_list_to_ht,
    import_blosum,
    import_grantham,
    prepare_pop_path_ht,
)

In [4]:
from bokeh.io import output_notebook, reset_output, show
from bokeh.models import Range1d

Defining Logger. This is mostly so that the functions imported from the repo have logging built-in. 

In [5]:
logging.basicConfig(
    format="%(asctime)s (%(name)s %(lineno)s): %(message)s",
    datefmt="%m/%d/%Y %I:%M:%S %p",
)
logger = logging.getLogger("MPC_experiment_notebook")
logger.setLevel(logging.INFO)

In [6]:
tmp_path = f"{TEMP_PATH_WITH_FAST_DEL}/mpc"

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import average_precision_score
from sklearn.model_selection import cross_val_score, GridSearchCV,StratifiedKFold
from xgboost import XGBClassifier

In [8]:
import joblib

In [9]:
# Don't truncate panda dataframe display
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# MPC modeling

In [10]:
context_ht = hl.read_table(
    "gs://gnomad/v4.1/constraint_coverage_corrected/preprocessed_data/gnomad.v4.1.context.preprocessed.ht"
).select_globals()

In [11]:
context_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'context': str 
    'vep': struct {
        most_severe_consequence: str, 
        transcript_consequences: array<struct {
            transcript_id: str, 
            gene_id: str, 
            gene_symbol: str, 
            biotype: str, 
            most_severe_consequence: str, 
            mane_select: str, 
            canonical: int32, 
            lof: str, 
            lof_flags: str, 
            sift_score: float64, 
            polyphen_score: float64, 
            domains: array<struct {
                db: str, 
                name: str
            }>, 
            uniprot_isoform: array<str>, 
            amino_acids: str, 
            codons: str
        }>
    } 
    'ref': str 
    'alt': str 
    'was_flipped': bool 
    'transition': bool 
    'cpg': bool 
    'mutation_type': str 
    'muta

In [12]:
context_ht = context_ht.select(transcript_consequences=context_ht.vep.transcript_consequences)

In [13]:
# Filter to loci in canonical constraint pass transcripts only
# and filter transcript consequences to those transcripts too
pass_context_ht = context_ht.annotate(
    transcript_consequences=context_ht.transcript_consequences.filter(
        lambda x: get_constraint_transcripts(filter_to_canonical=True, outlier=False).contains(x.transcript_id)
    )
)
pass_context_ht = pass_context_ht.filter(
    hl.len(pass_context_ht.transcript_consequences) > 0
)

WARNING (regional_missense_constraint_generic 630): Assumes LoF constraint has been separately calculated and that constraint HT exists...


In [14]:
pass_context_ht = pass_context_ht.checkpoint(
    f"{TEMP_PATH_WITH_SLOW_DEL}/mpc/pass_preprocessed_context.ht",
    _read_if_exists=True,
    overwrite=False,
)

In [15]:
def filter_vars_in_transcripts_exclusive(ht, keep_transcripts_expr):
    # Filter variant-level table
    # NOTE: Function re-keys `ht` to locus and alleles only (removes transcript from key)
    ht = ht.key_by("locus", "alleles")
    # Filter to retain variants only in transcripts to keep
    # and not in any other constraint QC pass transcripts including overlaps
    # This means that e.g. if there is a variant in the training transcripts and the test transcripts
    # (which are mutually exclusive), that variant will never be output from this function
    ht = ht.join(
        pass_context_ht.filter(
            pass_context_ht.transcript_consequences.all(
                lambda x: keep_transcripts_expr.contains(x.transcript_id)
            )
        ).select()
    )
    # Filter to remove variants in any other transcripts including overlaps
    ht = ht.anti_join(
        pass_context_ht.filter(
            pass_context_ht.transcript_consequences.any(
                lambda x: ~keep_transcripts_expr.contains(x.transcript_id)
            )
        ).select()
    )
    return ht

## Prepare truth set of variants for training and testing

Filter to benign (population) vs. pathogenic set and annotate labels.

**Current definitions:**

* Benign/population variant definition: Variants in gnomAD that have AF >0.1%.

* Pathogenic variant definition: Variants in ClinVar that are annotated as high-quality pathogenic for conditions that are severe.

    * "Severe" condition definition: Caused by gene predicted to be likely severely haploinsufficient as defined by pHaplo score being as severe as LOEUF-intolerant genes or G2P DD gene not LoF.

Note that `context_mpc_annots.ht` is missing AA-level annotations which are stored separately in `aa_metric_combined.ht`.

In [24]:
# Annotate variant-level features and AA-level features from separately stored tables
# Filter out transcript variants without all annotations in canonical transcripts
# variant-level annotations table
# NOTE: This function adds all transcript annotations for variants in multiple transcripts
# as `ht` must be keyed by locus and alleles
def annot_and_filt_features_for_mpc(ht, filter_all_annots: bool = True):        
    logger.info("Adding variant-level annotations...")
    # Add transcript-specific variant-level metrics + simultaneously filter out rows not in both
    # context annots table and input table with join
    context_mpc_annots_ht = hl.read_table(
        f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/context_mpc_annots.ht"
    )
    if list(ht.key) == ['locus', 'alleles']:
        logger.info("Keying annotations table by locus and alleles to join...")
        context_mpc_annots_ht = context_mpc_annots_ht._key_by_assert_sorted("locus", "alleles")
    ht = ht.join(context_mpc_annots_ht)
    
    logger.info("Adding AA-level annotations...")
    aa_annots_ht = hl.read_table(f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/aa_metric_combined.ht")
    ht = ht.annotate(**aa_annots_ht[ht.ref_aa, ht.alt_aa])
    
    if filter_all_annots:
        logger.info("Filtering to variants with all annotations...")
        ht = ht.filter(
            hl.all(
                [hl.is_defined(ht[x]) for x in list(ht.row_value)]
            )
        )
    
    logger.info("Deduplicating by key...")
    # Deduplicate - context annots table may have duplicate rows when keyng by locus, allele, transcript
    ht = ht.distinct()    

    return ht

In [25]:
# Filter to high coverage regions
def filt_ht_by_mc_region_median_an(ht, an_threshold = 90):
    median_an_all_mc_intervals_ht = hl.read_table(
        "gs://regional_missense_constraint/temp/all_rmc_intervals_median_AN.ht"
    )
    high_cov_median_an_all_mc_intervals_ht = median_an_all_mc_intervals_ht.filter(
        median_an_all_mc_intervals_ht.median_exomes_AN_percent >= an_threshold
    )
    # Explode to match on both locus and transcript
    expl_high_cov_median_an_all_mc_intervals_ht = explode_intervals_to_loci(
        high_cov_median_an_all_mc_intervals_ht,
        interval_field="interval", keep_intervals=True,
    ).key_by("locus", "transcript")
    # Filter to high coverage regions with median AN>=90% over loci used in RMC calculation
    ht = ht.filter(
        hl.is_defined(
            expl_high_cov_median_an_all_mc_intervals_ht[ht.locus, ht.transcript]
        )
    )
    return ht

In [26]:
def prepare_pop_path_w_benign_ht(
    gnomad_data_type: str = "exomes",
    af_threshold: float = 0.001,
    overwrite_temp: bool = False,
    overwrite_output: bool = True,
    freeze: int = CURRENT_FREEZE,
    adj_freq_index: int = 0,
    cov_threshold: int = 0,
    is_experiment: bool = True,
):
    # NOTE: This function does not filter to training transcripts - this step must be done downstream
    
    logger.info("Reading in ClinVar P/LP missense variants in severe HI plus DD non-LoF genes...")
    clinvar_ht = hl.read_table(
        "gs://regional_missense_constraint/resources/GRCh38/reference_data/ht/clinvar_pathogenic_missense_haplo_dd_nonlof.ht",
    )
    clinvar_ht = clinvar_ht.annotate(pop_v_path=0)
    
    logger.info("Reading in ClinVar B/LB missense variants in severe HI plus DD non-LoF genes...")    
    clinvar_blb_ht = hl.read_table(
        "gs://regional_missense_constraint/resources/GRCh38/reference_data/ht/clinvar_blb_mis.ht",
    )
    haplo_genes = hl.experimental.read_expression(
        'gs://regional_missense_constraint/resources/GRCh37/reference_data/ht/phaplo_genes.he'
    )
    dd_nonlof_genes = hl.experimental.read_expression(
        "gs://regional_missense_constraint/resources/GRCh38/reference_data/ht/dd_nonlof_genes.he"
    )
    clinvar_blb_ht = clinvar_blb_ht.annotate(
        gene=clinvar_blb_ht.info.GENEINFO.split(":")[0]
    )
    clinvar_blb_haplo_dd_nonlof_ht = clinvar_blb_ht.filter(
        haplo_genes.contains(clinvar_blb_ht.gene) |
        dd_nonlof_genes.contains(clinvar_blb_ht.gene)
    )
    clinvar_blb_haplo_dd_nonlof_ht = (
        clinvar_blb_haplo_dd_nonlof_ht
        .annotate(pop_v_path=1)
        .select("pop_v_path")
    )
    
    logger.info(
        "Importing gnomAD public data and filtering to high quality, common variants..."
    )
    gnomad_ht = get_gnomad_public_release(gnomad_data_type, adj_freq_index)
    gnomad_ht = gnomad_ht.filter(
        keep_criteria(
            ac_expr=gnomad_ht.ac,
            af_expr=gnomad_ht.af,
            filters_expr=gnomad_ht.filters,
            af_threshold=af_threshold,
            filter_to_rare=False,
        )
    )
    # NOTE: Filtering this HT to missense variants occurs when calling `annot_and_filt_features_for_mpc`
    # in order to avoid joining with the large context HT twice
    gnomad_ht = gnomad_ht.annotate(pop_v_path=1)
                
    # Remove any duplicates in ClinVar B/LB variants vs. common gnomAD before concatenating
    pop_benign_ht = gnomad_ht.select("pop_v_path").union(
        clinvar_blb_haplo_dd_nonlof_ht.anti_join(gnomad_ht).select("pop_v_path")
    )
        
    logger.info("Joining ClinVar and gnomAD HTs and filtering out overlaps...")
    ht = clinvar_ht.select("pop_v_path").union(pop_benign_ht.select("pop_v_path"))
    # Remove variants that are in both the benign and pathogenic set
    counts = ht.group_by(*ht.key).aggregate(n=hl.agg.count())
    counts = counts.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/clinvar_gnomad_benign_counts{'_experiment' if is_experiment else ''}.ht",
        _read_if_exists=not overwrite_temp,
        overwrite=overwrite_temp,
    )
    overlap = counts.filter(counts.n > 1)
    logger.info("%i ClinVar P/LP variants are in gnomAD common or benign", overlap.count())
    ht = ht.anti_join(overlap)
    ht = ht.checkpoint(
        f"{TEMP_PATH_WITH_SLOW_DEL}/joint_clinvar_gnomad_benign{'_experiment' if is_experiment else ''}.ht",
        _read_if_exists=not overwrite_temp,
        overwrite=overwrite_temp,
    )
    print("Variants after filtering out pop/path overlaps:")
    print(ht.aggregate(hl.agg.counter(ht.pop_v_path)))
    
    logger.info("Adding annotations and filtering to variants in canonical transcripts with all annotations...")
    ht = annot_and_filt_features_for_mpc(ht)
    ht = ht.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/joint_clinvar_gnomad_benign_transcript_annots{'_experiment' if is_experiment else ''}.ht",
        _read_if_exists=not overwrite_temp,
        overwrite=overwrite_temp,
    )
    print("Variants after annotating and filtering to canonical transcripts in variant annots table:")
    print(ht.aggregate(hl.agg.counter(ht.pop_v_path)))
    
    logger.info("Removing any variants in multiple transcripts after adding transcript-specific annotations...")
    counts = ht.group_by(*ht.key).aggregate(n=hl.agg.count())
    counts = counts.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/joint_clinvar_gnomad_benign_transcript_annots_counts{'_experiment' if is_experiment else ''}.ht",
        _read_if_exists=not overwrite_temp,
        overwrite=overwrite_temp,
    )
    overlap = counts.filter(counts.n > 1)
    logger.info("%i ClinVar P/LP or common gnomAD variants are in multiple transcripts", overlap.count())
    ht = ht.anti_join(overlap)
    ht = ht.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/joint_clinvar_gnomad_benign_single_transcript_annots{'_experiment' if is_experiment else ''}.ht",
        _read_if_exists=not overwrite_temp,
        overwrite=overwrite_temp,
    )
    print("Variants after removing variants in multiple transcripts:")
    print(ht.aggregate(hl.agg.counter(ht.pop_v_path)))
    
    # Filter pop and pathogenic variants to QC pass transcripts
    pass_transcripts = get_constraint_transcripts(filter_to_canonical=True, outlier=False)
    ht = ht.filter(pass_transcripts.contains(ht.transcript))
    
    # Filter pop and pathogenic variants to high coverage regions
    ht = filt_ht_by_mc_region_median_an(ht)
    
    ht = ht.checkpoint(
        f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{freeze}/{'experiment/' if is_experiment else ''}pop_common_benign_v_path_haplo_dd_nonlof.ht",
        _read_if_exists=not overwrite_output,
        overwrite=overwrite_output,
    )
    print("Variants after filtering to QC pass and high coverage regions:")
    print(ht.aggregate(hl.agg.counter(ht.pop_v_path)))

In [19]:
prepare_pop_path_w_benign_ht(is_experiment=False, overwrite_temp=True, overwrite_output=True)

INFO (MPC_experiment_notebook 13): Reading in ClinVar P/LP missense variants in severe HI plus DD non-LoF genes...
INFO (MPC_experiment_notebook 19): Reading in ClinVar B/LB missense variants in severe HI plus DD non-LoF genes...
INFO (MPC_experiment_notebook 42): Importing gnomAD public data and filtering to high quality, common variants...
INFO (MPC_experiment_notebook 66): Joining ClinVar and gnomAD HTs and filtering out overlaps...
INFO (MPC_experiment_notebook 79): 3 ClinVar P/LP variants are in gnomAD common or benign


Variants after filtering out pop/path overlaps:


INFO (MPC_experiment_notebook 91): Adding annotations and filtering to variants in canonical transcripts with all annotations...
INFO (MPC_experiment_notebook 7): Adding variant-level annotations...


{0: 22195, 1: 1477037}


INFO (MPC_experiment_notebook 15): Keying annotations table by locus and alleles to join...
INFO (MPC_experiment_notebook 19): Adding AA-level annotations...
INFO (MPC_experiment_notebook 24): Filtering to variants with all annotations...
INFO (MPC_experiment_notebook 31): Deduplicating by key...
2026-01-22 04:44:14.526 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-22 04:44:48.090 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-22 04:44:57.589 Hail: INFO: wrote table with 122699 rows in 151 partitions to gs://gnomad-tmp-4day/rmc/joint_clinvar_gnomad_benign_transcript_annots.ht


...

WARNING (regional_missense_constraint_generic 630): Assumes LoF constraint has been separately calculated and that constraint HT exists...


{0: 21389, 1: 101310}


2026-01-22 04:45:20.417 Hail: INFO: Coerced sorted dataset===>     (9 + 2) / 10]
2026-01-22 04:46:19.463 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-22 04:49:18.908 Hail: INFO: Coerced sorted dataset=====>(150 + 1) / 151]
2026-01-22 04:50:58.343 Hail: INFO: wrote table with 114569 rows in 151 partitions to gs://regional_missense_constraint/MPC/4.1/2/pop_common_benign_v_path_haplo_dd_nonlof.ht


Variants after filtering to QC pass and high coverage regions:


{0: 20931, 1: 93638}


## Set up training data

### Read in training data

In [20]:
# Include ClinVar benign variants with gnomAD common variants in training set
ht = hl.read_table(
    f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/pop_common_benign_v_path_haplo_dd_nonlof.ht"
)

In [21]:
ht.show()

,,,,,,,,,,,,,,,,,,,,,,
locus,alleles,pop_v_path,transcript,polyphen,phylop,gene_mis_oe,gene_mis_exp,gene_lof_oe,gene_lof_exp,section_mis_oe,section_mis_exp,upstream_section_mis_oe,upstream_section_mis_exp,downstream_section_mis_oe,downstream_section_mis_exp,ref_aa,alt_aa,blosum,grantham,aa_oe_overall,aa_oe_second_deriv,misbad
locus<GRCh38>,array<str>,int32,str,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str,str,float64,float64,float64,float64,float64
chr1:945122,"[""C"",""T""]",1,"""ENST00000327044""",0.00e+00,-3.20e+00,1.10e+00,9.80e+02,1.07e+00,9.42e+01,1.07e+00,5.77e+02,2.39e-01,1.67e+01,1.07e+00,5.77e+02,"""Arg""","""Gln""",1.00e+00,4.30e+01,1.01e+00,-1.20e-02,1.09e-01
chr1:946538,"[""G"",""A""]",1,"""ENST00000327044""",1.83e-01,4.86e+00,1.10e+00,9.80e+02,1.07e+00,9.42e+01,1.07e+00,5.77e+02,2.39e-01,1.67e+01,1.07e+00,5.77e+02,"""Ser""","""Leu""",-2.00e+00,1.45e+02,1.09e+00,-6.08e-03,3.59e-01
chr1:952419,"[""C"",""T""]",1,"""ENST00000327044""",8.83e-01,4.98e+00,1.10e+00,9.80e+02,1.07e+00,9.42e+01,1.07e+00,5.77e+02,2.39e-01,1.67e+01,1.07e+00,5.77e+02,"""Arg""","""His""",0.00e+00,2.90e+01,9.83e-01,-1.40e-02,9.76e-02
chr1:953858,"[""G"",""A""]",1,"""ENST00000327044""",8.80e-01,1.95e+00,1.10e+00,9.80e+02,1.07e+00,9.42e+01,1.18e+00,3.86e+02,1.18e+00,3.86e+02,2.39e-01,1.67e+01,"""Ala""","""Val""",0.00e+00,6.40e+01,1.01e+00,-5.14e-04,6.08e-01
chr1:956211,"[""C"",""T""]",1,"""ENST00000327044""",0.00e+00,-5.70e-02,1.10e+00,9.80e+02,1.07e+00,9.42e+01,1.18e+00,3.86e+02,1.18e+00,3.86e+02,2.39e-01,1.67e+01,"""Arg""","""His""",0.00e+00,2.90e+01,9.83e-01,-1.40e-02,9.76e-02
chr1:966542,"[""G"",""A""]",1,"""ENST00000379410""",9.29e-01,1.15e+00,1.21e+00,8.41e+02,9.34e-01,7.28e+01,1.21e+00,8.41e+02,1.21e+00,8.41e+02,1.21e+00,8.41e+02,"""Ser""","""Asn""",1.00e+00,4.60e+01,1.02e+00,-2.84e-03,4.96e-01
chr1:966542,"[""G"",""C""]",1,"""ENST00000379410""",7.36e-01,1.15e+00,1.21e+00,8.41e+02,9.34e-01,7.28e+01,1.21e+00,8.41e+02,1.21e+00,8.41e+02,1.21e+00,8.41e+02,"""Ser""","""Thr""",1.00e+00,5.80e+01,8.86e-01,-1.63e-03,5.57e-01
chr1:966543,"[""C"",""A""]",1,"""ENST00000379410""",9.31e-01,4.75e+00,1.21e+00,8.41e+02,9.34e-01,7.28e+01,1.21e+00,8.41e+02,1.21e+00,8.41e+02,1.21e+00,8.41e+02,"""Ser""","""Arg""",-1.00e+00,1.10e+02,9.41e-01,7.10e-03,1.00e+00


In [23]:
ht.aggregate(hl.agg.counter(ht.pop_v_path))

{0: 20931, 1: 93638}

### Select features for training and split into X and y pd DFs

In [21]:
features = [
    "polyphen", "phylop",
    "blosum", "grantham",
    "aa_oe_overall", "aa_oe_second_deriv", "misbad",
    "section_mis_oe", "section_mis_exp",
    "gene_mis_oe", "gene_mis_exp",
    "gene_lof_oe", "gene_lof_exp",
    "upstream_section_mis_oe", "upstream_section_mis_exp",
    "downstream_section_mis_oe", "downstream_section_mis_exp",
]

In [22]:
len(features)

17

In [23]:
select_ht = ht.select(*(features + ["pop_v_path"]))

### In training set - not final model

In [25]:
train_ht = filter_vars_in_transcripts_exclusive(
    select_ht,
    hl.experimental.read_expression(train_val_test_transcripts_path())
)

In [26]:
train_df = train_ht.to_pandas().drop(["locus", "alleles"], axis=1)

In [27]:
train_X = train_df.drop("pop_v_path", axis=1)

In [28]:
train_df.head()

,polyphen,phylop,blosum,grantham,aa_oe_overall,aa_oe_second_deriv,misbad,section_mis_oe,section_mis_exp,gene_mis_oe,gene_mis_exp,gene_lof_oe,gene_lof_exp,upstream_section_mis_oe,upstream_section_mis_exp,downstream_section_mis_oe,downstream_section_mis_exp,pop_v_path
0,0.929,1.149,1.0,46.0,1.024215,-0.002841,0.495691,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53,1
1,0.736,1.149,1.0,58.0,0.885923,-0.001635,0.557404,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53,1
2,0.931,4.747,-1.0,110.0,0.940523,0.007103,1.0,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53,1
3,0.018,2.024,0.0,64.0,1.013,-0.000514,0.607726,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53,1
4,0.06,-2.845,2.0,45.0,0.842918,-0.000032,0.661406,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53,1


In [29]:
train_y = train_df["pop_v_path"]

In [30]:
train_df.shape

(89355, 18)

## Logistic regression

In [31]:
# Logistic regression
lr_clf = LogisticRegressionCV(
    penalty='l1',
    solver='liblinear',
    cv=5,
    random_state=42,
    scoring='average_precision'
)

In [32]:
lr_clf.fit(train_X, train_y)

,Cs,10
,fit_intercept,True
,cv,5
,dual,False
,penalty,'l1'
,scoring,'average_precision'
,solver,'liblinear'
,tol,0.0001
,max_iter,100
,class_weight,None
,n_jobs,None


In [24]:
overwrite = False
lr_clf_path  = f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/train/pop_common_benign_lr_clf.pkl"
if file_exists(lr_clf_path) and not overwrite:
    print("Reading model...")
    with hl.hadoop_open(lr_clf_path, "rb") as p:
        lr_clf = joblib.load(p)
else:
    print("Saving model...")
    with hl.hadoop_open(lr_clf_path, "wb") as p:
        joblib.dump(lr_clf, p, compress=3)

Reading model...


## Xgboost

### Feature selection with Recursive Feature Elimination

In [42]:
xgb_clf = XGBClassifier(
    random_state=42,
    objective='binary:logistic',
    eval_metric='aucpr',
)

In [43]:
xgb_rfecv = RFECV(
    estimator=xgb_clf, 
    cv=StratifiedKFold(n_splits=5),
    scoring='average_precision',
    n_jobs=-1
)

In [44]:
xgb_rfecv.fit(train_X, train_y)

,estimator,"XGBClassifier...ree=None, ...)"
,step,1
,min_features_to_select,1
,cv,StratifiedKFo...shuffle=False)
,scoring,'average_precision'
,verbose,0
,n_jobs,-1
,importance_getter,'auto'
,objective,'binary:logistic'
,base_score,None
,booster,None


In [45]:
print(f"Optimal number of features: {xgb_rfecv.n_features_}")
print(f"Selected features mask: {xgb_rfecv.support_}")
print(f"Feature rankings: {xgb_rfecv.ranking_}")

Optimal number of features: 12
Selected features mask: [ True  True False False False  True  True  True  True  True  True  True
  True False  True  True False]
Feature rankings: [1 1 6 5 4 1 1 1 1 1 1 1 1 3 1 1 2]


In [48]:
train_X.head()

,polyphen,phylop,blosum,grantham,aa_oe_overall,aa_oe_second_deriv,misbad,section_mis_oe,section_mis_exp,gene_mis_oe,gene_mis_exp,gene_lof_oe,gene_lof_exp,upstream_section_mis_oe,upstream_section_mis_exp,downstream_section_mis_oe,downstream_section_mis_exp
0,0.929,1.149,1.0,46.0,1.024215,-0.002841,0.495691,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53
1,0.736,1.149,1.0,58.0,0.885923,-0.001635,0.557404,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53
2,0.931,4.747,-1.0,110.0,0.940523,0.007103,1.0,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53
3,0.018,2.024,0.0,64.0,1.013,-0.000514,0.607726,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53
4,0.06,-2.845,2.0,45.0,0.842918,-0.000032,0.661406,1.2076,840.53,1.207565,840.534331,0.933842,72.817419,1.2076,840.53,1.2076,840.53


In [25]:
overwrite = False
xgb_rfecv_path  = f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/train/pop_common_benign_xgb_rfecv.pkl"
if file_exists(xgb_rfecv_path) and not overwrite:
    print("Reading model...")
    with hl.hadoop_open(xgb_rfecv_path, "rb") as p:
        xgb_rfecv = joblib.load(p)
else:
    print("Saving model...")
    with hl.hadoop_open(xgb_rfecv_path, "wb") as p:
        joblib.dump(xgb_rfecv, p, compress=3)

Reading model...


## Generate final models with selected hyperparameters on all data

In [26]:
final_train_df = select_ht.to_pandas().drop(["locus", "alleles"], axis=1)

In [27]:
final_train_X = final_train_df.drop("pop_v_path", axis=1)

In [28]:
final_train_df.head()

,polyphen,phylop,blosum,grantham,aa_oe_overall,aa_oe_second_deriv,misbad,section_mis_oe,section_mis_exp,gene_mis_oe,gene_mis_exp,gene_lof_oe,gene_lof_exp,upstream_section_mis_oe,upstream_section_mis_exp,downstream_section_mis_oe,downstream_section_mis_exp,pop_v_path
0,0.0,-3.204,1.0,43.0,1.009231,-0.011974,0.109093,1.0652,577.38,1.097804,980.138924,1.07193,94.222587,0.23894,16.74,1.0652,577.38,1
1,0.183,4.855,-2.0,145.0,1.091555,-0.006078,0.358737,1.0652,577.38,1.097804,980.138924,1.07193,94.222587,0.23894,16.74,1.0652,577.38,1
2,0.883,4.976,0.0,29.0,0.983269,-0.013979,0.097633,1.0652,577.38,1.097804,980.138924,1.07193,94.222587,0.23894,16.74,1.0652,577.38,1
3,0.88,1.948,0.0,64.0,1.013,-0.000514,0.607726,1.1839,386.02,1.097804,980.138924,1.07193,94.222587,1.1839,386.02,0.23894,16.74,1
4,0.0,-0.057,0.0,29.0,0.983269,-0.013979,0.097633,1.1839,386.02,1.097804,980.138924,1.07193,94.222587,1.1839,386.02,0.23894,16.74,1


In [29]:
final_train_y = final_train_df["pop_v_path"]

In [30]:
final_train_df.shape

(114569, 18)

Keep same LR hyperparameters — `C` and features set to 0 (eliminated).

In [26]:
model_support_masks = {
    "lr": ~np.isclose(lr_clf.coef_[0], 0.0, atol=1e-8),
    "xgb": xgb_rfecv.support_,
}

In [27]:
model_support_masks

{'lr': array([ True,  True,  True,  True,  True, False,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True]),
 'xgb': array([ True,  True, False, False, False,  True,  True,  True,  True,
         True,  True,  True,  True, False,  True,  True, False])}

In [33]:
final_lr = LogisticRegression(
    C=lr_clf.C_[0],
    penalty='l1',
    solver='liblinear',
    random_state=42,
)

In [71]:
final_lr.fit(
    final_train_X.loc[:, model_support_masks["lr"]],
    final_train_y,
)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.3593813663804626
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [28]:
overwrite = False
final_lr_path  = f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/pop_common_benign_lr.pkl"
if file_exists(final_lr_path) and not overwrite:
    print("Reading model...")
    with hl.hadoop_open(final_lr_path, "rb") as p:
        final_lr = joblib.load(p)
else:
    print("Saving model...")
    with hl.hadoop_open(final_lr_path, "wb") as p:
        joblib.dump(final_lr, p, compress=3)

Reading model...


Keep features chosen in XGB RFE.

In [109]:
final_xgb = XGBClassifier(
    random_state=42,
    objective='binary:logistic',
    eval_metric='aucpr',
)

In [110]:
final_xgb.fit(
    final_train_X.loc[:, model_support_masks["xgb"]],
    final_train_y,
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'aucpr'


In [29]:
overwrite = False
final_xgb_path  = f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/pop_common_benign_xgb.pkl"
if file_exists(final_xgb_path) and not overwrite:
    print("Reading model...")
    with hl.hadoop_open(final_xgb_path, "rb") as p:
        final_xgb = joblib.load(p)
else:
    print("Saving model...")
    with hl.hadoop_open(final_xgb_path, "wb") as p:
        joblib.dump(final_xgb, p, compress=3)

Reading model...


# Run MPC transformation on each model

## Get context table with all annotations

In [30]:
context_oe_ht = context_with_oe.ht()

In [31]:
context_oe_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'polyphen': struct {
        prediction: str, 
        score: float64
    } 
    'sift': struct {
        prediction: str, 
        score: float64
    } 
    'codons': str 
    'most_severe_consequence': str 
    'transcript': str 
    'ref': str 
    'alt': str 
    'oe': float64 
----------------------------------------
Key: ['locus', 'alleles', 'transcript']
----------------------------------------


In [32]:
context_oe_ht.show()

+---------------+------------+---------------------+----------------+
| locus         | alleles    | polyphen.prediction | polyphen.score |
+---------------+------------+---------------------+----------------+
| locus<GRCh38> | array<str> | str                 |        float64 |
+---------------+------------+---------------------+----------------+
| chr1:65568    | ["A","C"]  | "benign"            |       2.50e-02 |
| chr1:65568    | ["A","G"]  | "benign"            |       7.00e-03 |
| chr1:65569    | ["A","C"]  | "benign"            |       7.00e-03 |
| chr1:65569    | ["A","G"]  | "benign"            |       1.20e-02 |
| chr1:65569    | ["A","T"]  | "benign"            |       1.52e-01 |
| chr1:65570    | ["G","C"]  | "benign"            |       7.00e-03 |
| chr1:65570    | ["G","T"]  | "benign"            |       7.00e-03 |
| chr1:65571    | ["A","C"]  | "benign"            |       0.00e+00 |
| chr1:65571    | ["A","G"]  | "benign"            |       0.00e+00 |
| chr1:65572    | ["A","C"]  | "benign"            |       3.30e-02 |
+---------------+------------+---------------------+----------------+

+------------------------------+------------+-----------+
| sift.prediction              | sift.score | codons    |
+------------------------------+------------+-----------+
| str                          |    float64 | str       |
+------------------------------+------------+-----------+
| "tolerated_low_confidence"   |   3.70e-01 | "Aag/Cag" |
| "tolerated_low_confidence"   |   3.30e-01 | "Aag/Gag" |
| "tolerated_low_confidence"   |   2.50e-01 | "aAg/aCg" |
| "tolerated_low_confidence"   |   8.20e-01 | "aAg/aGg" |
| "tolerated_low_confidence"   |   6.00e-02 | "aAg/aTg" |
| "tolerated_low_confidence"   |   2.20e-01 | "aaG/aaC" |
| "tolerated_low_confidence"   |   2.20e-01 | "aaG/aaT" |
| "tolerated_low_confidence"   |   9.30e-01 | "Aag/Cag" |
| "deleterious_low_confidence" |   4.00e-02 | "Aag/Gag" |
| "deleterious_low_confidence" |   3.00e-02 | "aAg/aCg" |
+------------------------------+------------+-----------+

+-------------------------+-------------------+-------+-------+----------+
| most_severe_consequence | transcript        | ref   | alt   |       oe |
+-------------------------+-------------------+-------+-------+----------+
| str                     | str               | str   | str   |  float64 |
+-------------------------+-------------------+-------+-------+----------+
| "missense_variant"      | "ENST00000641515" | "Lys" | "Gln" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Glu" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Thr" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Arg" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Met" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Asn" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Asn" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Gln" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Glu" | 0.00e+00 |
| "missense_variant"      | "ENST00000641515" | "Lys" | "Thr" | 0.00e+00 |
+-------------------------+-------------------+-------+-------+----------+
showing top 10 rows

In [33]:
context_oe_ht.count()

74082272

In [34]:
context_mpc_annots_ht = hl.read_table(
    f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/context_mpc_annots.ht"
)

## Begin generating MPC

In [36]:
annot_context_oe_ht = annot_and_filt_features_for_mpc(context_oe_ht.select())

INFO (MPC_experiment_notebook 7): Adding variant-level annotations...
INFO (MPC_experiment_notebook 19): Adding AA-level annotations...
INFO (MPC_experiment_notebook 24): Filtering to variants with all annotations...
INFO (MPC_experiment_notebook 31): Deduplicating by key...


In [37]:
annot_context_oe_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'polyphen': float64 
    'phylop': float64 
    'gene_mis_oe': float64 
    'gene_mis_exp': float64 
    'gene_lof_oe': float64 
    'gene_lof_exp': float64 
    'section_mis_oe': float64 
    'section_mis_exp': float64 
    'upstream_section_mis_oe': float64 
    'upstream_section_mis_exp': float64 
    'downstream_section_mis_oe': float64 
    'downstream_section_mis_exp': float64 
    'ref_aa': str 
    'alt_aa': str 
    'blosum': float64 
    'grantham': float64 
    'aa_oe_overall': float64 
    'aa_oe_second_deriv': float64 
    'misbad': float64 
----------------------------------------
Key: ['locus', 'alleles', 'transcript']
----------------------------------------


In [38]:
annot_context_oe_ht = annot_context_oe_ht.select(*features)

In [39]:
annot_context_oe_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'polyphen': float64 
    'phylop': float64 
    'blosum': float64 
    'grantham': float64 
    'aa_oe_overall': float64 
    'aa_oe_second_deriv': float64 
    'misbad': float64 
    'section_mis_oe': float64 
    'section_mis_exp': float64 
    'gene_mis_oe': float64 
    'gene_mis_exp': float64 
    'gene_lof_oe': float64 
    'gene_lof_exp': float64 
    'upstream_section_mis_oe': float64 
    'upstream_section_mis_exp': float64 
    'downstream_section_mis_oe': float64 
    'downstream_section_mis_exp': float64 
----------------------------------------
Key: ['locus', 'alleles', 'transcript']
----------------------------------------


In [40]:
annot_context_oe_ht = annot_context_oe_ht.checkpoint(
    f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/context_oe_mpc_annot_addtl_oe_exp_filt.ht",
    _read_if_exists=True,
    overwrite=False,
)

In [41]:
models = {"lr": final_lr, "xgb": final_xgb}

In [22]:
model_types = list(models.keys())

In [42]:
# Chunk table to make this feasible
chunk_size = 600000

In [43]:
head_size = chunk_size

In [44]:
n_rows = annot_context_oe_ht.count()

In [45]:
n_rows

70313598

In [46]:
n_rows / chunk_size

117.18933

In [47]:
model_support_masks

{'lr': array([ True,  True,  True,  True,  True, False,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True]),
 'xgb': array([ True,  True, False, False, False,  True,  True,  True,  True,
         True,  True,  True,  True, False,  True,  True, False])}

In [48]:
overwrite = False
while head_size < (n_rows + chunk_size):
    i = int(head_size / chunk_size)
    print(i)
    # Extract table chunk
    chunk_ht_path = f"{TEMP_PATH_WITH_SLOW_DEL}/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk{i}.ht"
    if file_exists(chunk_ht_path) and not overwrite:
        chunk_ht = hl.read_table(chunk_ht_path)
    else:
        chunk_ht = annot_context_oe_ht.head(head_size).tail(chunk_size)
        # Index chunk table to match to fitted scores table later
        chunk_ht = chunk_ht.add_index()

        # Extract features for predicting
        chunk_all_missense_X = chunk_ht._key_by_assert_sorted().drop(
            "locus", "alleles", "transcript", "idx"
        ).to_pandas()

        for classifier_type, model in models.items():
            X = chunk_all_missense_X.loc[:, model_support_masks[classifier_type]]
            if i == 1:
                print(classifier_type)
                display(X.head())
            fitted_scores = model.predict_proba(X)[:, 0]
            fitted_scores_ht = hl.Table.from_pandas(pd.DataFrame(fitted_scores, columns=["score"]))
            fitted_scores_ht = fitted_scores_ht.add_index().key_by("idx")
            chunk_ht = chunk_ht.annotate(
                **{classifier_type: fitted_scores_ht[chunk_ht.idx].score}
            )
        # Write out
        chunk_ht = chunk_ht.checkpoint(
            chunk_ht_path,
            _read_if_exists=not overwrite,
            overwrite=overwrite,
        )

    # Increment counter for next iteration
    head_size = head_size + chunk_size

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67


2026-01-22 20:04:29.286 Hail: INFO: Coerced sorted dataset          (1 + 1) / 2]
2026-01-22 20:04:34.043 Hail: INFO: Coerced sorted dataset        (12 + 4) / 16]
2026-01-22 20:04:36.487 Hail: INFO: Coerced sorted dataset          (1 + 1) / 2]
2026-01-22 20:04:40.346 Hail: INFO: Coerced sorted dataset        (13 + 3) / 16]
2026-01-22 20:04:43.617 Hail: INFO: Coerced sorted dataset          (1 + 1) / 2]
2026-01-22 20:04:49.212 Hail: INFO: wrote table with 600000 rows in 2 partitions to gs://gnomad-tmp/rmc/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk67.ht


68


2026-01-22 20:07:20.045 Hail: INFO: Coerced sorted dataset          (2 + 1) / 3]
2026-01-22 20:07:23.905 Hail: INFO: Coerced sorted dataset        (10 + 4) / 16]
2026-01-22 20:07:25.950 Hail: INFO: Coerced sorted dataset          (2 + 1) / 3]
2026-01-22 20:07:29.612 Hail: INFO: Coerced sorted dataset        (12 + 4) / 16]
2026-01-22 20:07:32.790 Hail: INFO: Coerced sorted dataset          (2 + 1) / 3]
2026-01-22 20:07:37.831 Hail: INFO: wrote table with 600000 rows in 3 partitions to gs://gnomad-tmp/rmc/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk68.ht


69


...

2026-01-22 22:18:38.177 Hail: INFO: Coerced sorted dataset          (1 + 1) / 2]
2026-01-22 22:18:41.587 Hail: INFO: Coerced sorted dataset
2026-01-22 22:18:43.441 Hail: INFO: Coerced sorted dataset          (1 + 1) / 2]
2026-01-22 22:18:46.587 Hail: INFO: Coerced sorted dataset
2026-01-22 22:18:49.713 Hail: INFO: Coerced sorted dataset          (1 + 1) / 2]
2026-01-22 22:18:54.367 Hail: INFO: wrote table with 600000 rows in 2 partitions to gs://gnomad-tmp/rmc/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk116.ht


117


2026-01-22 22:21:19.664 Hail: INFO: Coerced sorted dataset          (2 + 1) / 3]
2026-01-22 22:21:23.047 Hail: INFO: Coerced sorted dataset
2026-01-22 22:21:24.796 Hail: INFO: Coerced sorted dataset          (2 + 1) / 3]
2026-01-22 22:21:28.053 Hail: INFO: Coerced sorted dataset=>      (14 + 2) / 16]
2026-01-22 22:21:30.526 Hail: INFO: Coerced sorted dataset          (2 + 1) / 3]
2026-01-22 22:21:35.157 Hail: INFO: wrote table with 600000 rows in 3 partitions to gs://gnomad-tmp/rmc/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk117.ht


118


2026-01-22 22:24:00.518 Hail: INFO: Coerced sorted dataset          (0 + 2) / 2]
2026-01-22 22:24:03.954 Hail: INFO: Coerced sorted dataset
2026-01-22 22:24:05.424 Hail: INFO: Coerced sorted dataset          (0 + 2) / 2]
2026-01-22 22:24:08.693 Hail: INFO: Coerced sorted dataset
2026-01-22 22:24:10.855 Hail: INFO: Coerced sorted dataset          (0 + 2) / 2]
2026-01-22 22:24:14.728 Hail: INFO: wrote table with 600000 rows in 2 partitions to gs://gnomad-tmp/rmc/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk118.ht


In [49]:
from math import ceil

In [53]:
# Concatenate chunked tables and checkpoint as fitted scores HT
start_i = 1
context_oe_fitted_scores_ht = hl.read_table(
    f"{TEMP_PATH_WITH_SLOW_DEL}/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk{start_i}.ht"
).select(*model_types)
# Join up to the second to last chunk
n_chunks = ceil(annot_context_oe_ht.count() / chunk_size)
for i in np.arange(2, n_chunks + 1):
    context_oe_fitted_scores_ht = context_oe_fitted_scores_ht.union(
        hl.read_table(
            f"{TEMP_PATH_WITH_SLOW_DEL}/mpc/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_chunk{i}.ht"
        )
        .select(*model_types)
    )
# Last chunk may overlap with second to last if table size not a multiple of chunk size,
# so remove any overlap with distinct()
context_oe_fitted_scores_ht = context_oe_fitted_scores_ht.distinct()

In [54]:
context_oe_fitted_scores_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'lr': float64 
    'xgb': float64 
----------------------------------------
Key: ['locus', 'alleles', 'transcript']
----------------------------------------


In [55]:
# Fix fitted scores to be 1-proba (i.e. probability of class 1)
# so that fitted scores are smaller for higher deleteriousness (class 0)
context_oe_fitted_scores_ht = context_oe_fitted_scores_ht.annotate(
    lr=1-context_oe_fitted_scores_ht.lr,
    xgb=1-context_oe_fitted_scores_ht.xgb,
)

In [56]:
context_oe_fitted_scores_ht = context_oe_fitted_scores_ht.repartition(1000)
context_oe_fitted_scores_ht = context_oe_fitted_scores_ht.checkpoint(
    f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_pop_common_benign_addtl_oe_exp.ht",
    _read_if_exists=False,
    overwrite=True,
)

2026-01-22 22:40:31.182 Hail: INFO: wrote table with 70800000 rows in 1000 partitions to gs://regional_missense_constraint/MPC/4.1/2/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_pop_common_benign_addtl_oe_exp.ht


In [18]:
context_oe_fitted_scores_ht = hl.read_table(
    f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/context_oe_mpc_annot_filt_all_transcripts_fitted_scores_pop_common_benign_addtl_oe_exp.ht"
)

In [19]:
context_oe_fitted_scores_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'lr': float64 
    'xgb': float64 
----------------------------------------
Key: ['locus', 'alleles', 'transcript']
----------------------------------------


In [21]:
context_oe_fitted_scores_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'lr': float64 
    'xgb': float64 
----------------------------------------
Key: ['locus', 'alleles', 'transcript']
----------------------------------------


In [23]:
context_oe_fitted_scores_ht.aggregate(
    hl.struct(
        **{
            model: hl.agg.stats(context_oe_fitted_scores_ht[model])
            for model in model_types
        }
    )
)

Struct(lr=Struct(mean=0.8122628329907238, stdev=0.26516274875321144, min=0.0003833419806297256, max=1.0, n=70313598, sum=57113122.30925089), xgb=Struct(mean=0.8792936990809292, stdev=0.244635611472095, min=6.318092346191406e-05, max=0.9999996423721313, n=70313598, sum=61826303.68110943))

In [57]:
context_oe_fitted_scores_ht.aggregate(
    hl.struct(
        **{
            model: hl.agg.stats(context_oe_fitted_scores_ht[model])
            for model in list(models.keys())
        }
    )
)

Struct(lr=Struct(mean=0.8118447898119299, stdev=0.2655702377779861, min=0.0003833419806297256, max=1.0, n=70800000, sum=57478611.118684635), xgb=Struct(mean=0.8788744760047287, stdev=0.24507963545338812, min=6.318092346191406e-05, max=0.9999996423721313, n=70800000, sum=62224312.90113479))

In [28]:
# Filter to gnomAD common + benign transcript variants
joint_clinvar_gnomad_ht = hl.read_table(
    f"{TEMP_PATH_WITH_SLOW_DEL}/joint_clinvar_gnomad_benign.ht"
)
nonpath_common_gnomad_ht = joint_clinvar_gnomad_ht.filter(joint_clinvar_gnomad_ht.pop_v_path == 1)

In [29]:
nonpath_common_gnomad_ht.describe()

----------------------------------------
Global fields:
    'vep_version': str 
    'vep_help': str 
    'vep_config': str 
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'pop_v_path': int32 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [30]:
nonpath_common_gnomad_ht.count()

1477037

In [31]:
nonpath_common_gnomad_fitted_scores_ht = (
    context_oe_fitted_scores_ht
    ._key_by_assert_sorted("locus", "alleles")
    .join(nonpath_common_gnomad_ht)
)

In [32]:
nonpath_common_gnomad_fitted_scores_ht.count()

101599

In [33]:
nonpath_common_gnomad_fitted_scores_ht.distinct().count()

101310

In [34]:
nonpath_common_gnomad_fitted_scores_ht.describe()

----------------------------------------
Global fields:
    'vep_version': str 
    'vep_help': str 
    'vep_config': str 
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'transcript': str 
    'lr': float64 
    'xgb': float64 
    'pop_v_path': int32 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


About 1% of variants in the gnomAD common or benign set are in multiple transcripts. Since we no longer filter to training transcripts only with `filter_exclusive`, we should deduplicate these variants. We will just select the first transcript for each variant.

In [35]:
nonpath_common_gnomad_fitted_scores_ht = nonpath_common_gnomad_fitted_scores_ht.order_by(
    "locus", "alleles", "transcript"
)

In [36]:
nonpath_common_gnomad_fitted_scores_ht = (
    nonpath_common_gnomad_fitted_scores_ht
    .group_by("locus", "alleles")
    .aggregate(
        pop_v_path=hl.agg.take(nonpath_common_gnomad_fitted_scores_ht.pop_v_path, 1)[0],
        lr=hl.agg.take(nonpath_common_gnomad_fitted_scores_ht.lr, 1)[0],
        xgb=hl.agg.take(nonpath_common_gnomad_fitted_scores_ht.xgb, 1)[0],
    )
)

In [72]:
nonpath_common_gnomad_fitted_scores_ht = nonpath_common_gnomad_fitted_scores_ht.naive_coalesce(100)
nonpath_common_gnomad_fitted_scores_ht = (
    nonpath_common_gnomad_fitted_scores_ht.checkpoint(
        f"{TEMP_PATH_WITH_SLOW_DEL}/mpc/nonpath_common_gnomad_fitted_scores.ht",
        _read_if_exists=False,
        overwrite=True,
    )
)

2026-01-22 22:54:58.666 Hail: INFO: Coerced sorted dataset======>(99 + 1) / 100]
2026-01-22 22:57:04.291 Hail: INFO: Coerced sorted dataset======>(99 + 1) / 100]
2026-01-22 22:58:18.246 Hail: INFO: wrote table with 101310 rows in 100 partitions to gs://gnomad-tmp/rmc/mpc/nonpath_common_gnomad_fitted_scores.ht


In [38]:
nonpath_common_gnomad_fitted_scores_ht.show()

,,,,
locus,alleles,pop_v_path,lr,xgb
locus<GRCh38>,array<str>,int32,float64,float64
chr1:69428,"[""T"",""G""]",1,5.78e-02,9.94e-01
chr1:924483,"[""C"",""A""]",1,9.98e-01,4.90e-01
chr1:924499,"[""C"",""A""]",1,9.96e-01,4.21e-01
chr1:924523,"[""C"",""A""]",1,9.93e-01,6.60e-01
chr1:924567,"[""C"",""A""]",1,9.98e-01,4.64e-01
chr1:924673,"[""C"",""A""]",1,1.00e+00,9.83e-01
chr1:930248,"[""G"",""A""]",1,1.00e+00,9.15e-01
chr1:930314,"[""C"",""T""]",1,1.00e+00,9.96e-01


In [43]:
nonpath_common_gnomad_fitted_scores_ht.describe()

----------------------------------------
Global fields:
    'vep_version': str 
    'vep_help': str 
    'vep_config': str 
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'pop_v_path': int32 
    'lr': float64 
    'xgb': float64 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [75]:
nonpath_common_gnomad_fitted_scores_ht.aggregate(
    hl.struct(
        **{
            model: hl.agg.stats(nonpath_common_gnomad_fitted_scores_ht[model])
            for model in model_types
        }
    )
)

Struct(lr=Struct(mean=0.937184127611849, stdev=0.14732690183390607, min=0.0020626470088505044, max=1.0, n=101310, sum=94946.12396835642), xgb=Struct(mean=0.970637018188015, stdev=0.09810457789258938, min=0.0035377144813537598, max=0.9999991655349731, n=101310, sum=98335.23631262779))

In [40]:
nonpath_common_gnomad_fitted_scores_ht.aggregate(
    hl.struct(
        **{
            model: hl.agg.count_where(
                hl.is_missing(nonpath_common_gnomad_fitted_scores_ht[model])
            )
            for model in model_types
        }
    )
)

Struct(lr=0, xgb=0)

In [71]:
nonpath_common_gnomad_fitted_scores_ht.describe()

----------------------------------------
Global fields:
    'vep_version': str 
    'vep_help': str 
    'vep_config': str 
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'pop_v_path': int32 
    'lr': float64 
    'xgb': float64 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [72]:
nonpath_common_gnomad_fitted_scores_ht.aggregate(
    hl.agg.max(nonpath_common_gnomad_fitted_scores_ht.xgb)
)

0.9999991655349731

In [51]:
# NOTE: If there is 1 gnomAD common variant with a fitted score less than
# the fitted score for a given variant, that variant will get an MPC score of 5.00565
# We will set n_less_eq0_float to give MPC = 6 for a variant with
# a more extreme fitted score than all gnomAD common variants
n_less_eq0_float = nonpath_common_gnomad_fitted_scores_ht.count() * 10**-6
print(n_less_eq0_float)

0.10131


In [65]:
# Calculate n_less and MPC transformation for train context fitted scores
for model in model_types:
    print(model)
    gnomad_ht = (
        nonpath_common_gnomad_fitted_scores_ht
        .select(model)
        .rename({model: "fitted_score"})
    )
    gnomad_ht = gnomad_ht.group_by("fitted_score").aggregate(n_var=hl.agg.count())
    gnomad_ht = gnomad_ht.order_by("fitted_score")
    gnomad_ht = gnomad_ht.key_by("fitted_score")
    gnomad_ht = gnomad_ht.annotate(n_less=hl.scan.sum(gnomad_ht.n_var))
    # Make n_less a non-zero value if it is zero
    gnomad_ht = gnomad_ht.annotate(
        n_less=hl.if_else(
            gnomad_ht.n_less == 0,
            n_less_eq0_float,
            gnomad_ht.n_less,
        )
    )
    # Add index annotation to table and convert from int64
    # (default value returned by `add_index`) to int32
    # This is necessary for code in `annotate_mpc` downstream
    # (`annotate_mpc will try to join this index field with an int32 field;
    # `hl.binary_search` returns an int32 by default)
    gnomad_ht = gnomad_ht.add_index()
    gnomad_ht = gnomad_ht.annotate(idx=hl.int(gnomad_ht.idx))
    gnomad_ht = gnomad_ht.key_by("idx")
    gnomad_ht = gnomad_ht.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/mpc/nonpath_common_gnomad_fitted_scores_grouped_{model}.ht",
        _read_if_exists=False,
        overwrite=True,
    )
    print("N rows in common gnomAD grouped fitted score table:")
    print(gnomad_ht.count())
    
    gnomad_scores = gnomad_ht.aggregate(
        hl.sorted(hl.agg.collect(gnomad_ht.fitted_score))
    )
    gnomad_scores_len = len(gnomad_scores)

    # Get total number of gnomAD common variants
    gnomad_var_count = nonpath_common_gnomad_fitted_scores_ht.count()
    print("Number of total gnomAD common variants:")
    print(gnomad_var_count)

    logger.info("Getting n_less values for input variants...")
    # Annotate HT with sorted array of gnomAD fitted scores
    scores_ht = (
        context_oe_fitted_scores_ht
        .select(model)
        .rename({model: "fitted_score"})
    )
    scores_ht = scores_ht.annotate_globals(gnomad_scores=gnomad_scores)
    # Checkpoint here to force the gnomAD join to complete
    scores_ht = scores_ht.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/mpc/context_fitted_and_gnomad_scores_{model}.ht",
        _read_if_exists=False,
        overwrite=True,
    )
    
    # Search all gnomAD scores to find first score that is
    # greater than or equal to score to be annotated
    # `binary_search` will return the index of the first gnomAD fitted score that
    # is >= the score of interest
    # e.g., if the score of interest is 0.45, and gnomAD fitted scores are
    # [0.3, 0.4, 0.5], then `binary_search` will return an index of 2
    # the `n_less` of 0.5 will contain the counts of variants with gnomAD scores of
    # 0.3 and 0.4 due to the non-inclusive nature of scans
    # (n_less[0.5] = n_var[0.3] + n_var[0.4])
    scores_ht = scores_ht.annotate(
        idx=hl.binary_search(scores_ht.gnomad_scores, scores_ht.fitted_score)
    )
    scores_ht = scores_ht.annotate(
        n_less=hl.if_else(
            # Make n_less equal to total gnomAD common variant count if
            # index is equal to the length of the gnomAD scores array
            scores_ht.idx == gnomad_scores_len,
            gnomad_var_count,
            gnomad_ht[scores_ht.idx].n_less,
        )
    )
    # Checkpoint here to force the binary search to compute
    scores_ht = scores_ht.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/mpc/context_binary_{model}.ht",
        _read_if_exists=False,
        overwrite=True,
    )

    logger.info("Calculating MPC scores...")
    scores_ht = scores_ht.annotate(
        mpc=-(hl.log10(scores_ht.n_less / gnomad_var_count))
    )
    scores_ht = scores_ht.checkpoint(
        f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/mpc_{model}_common_benign_addtl_oe_exp.ht",
        _read_if_exists=False,
        overwrite=True,
    )

lr


2026-02-10 22:01:35.009 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-02-10 22:01:44.004 Hail: INFO: Coerced sorted dataset==>    (91 + 9) / 100]
2026-02-10 22:01:51.054 Hail: INFO: wrote table with 101181 rows in 100 partitions to gs://gnomad-tmp-4day/rmc/mpc/nonpath_common_gnomad_fitted_scores_grouped_lr.ht


N rows in common gnomAD grouped fitted score table:
101181


INFO (MPC_experiment_notebook 47): Getting n_less values for input variants...


Number of total gnomAD common variants:
101310


...

2026-02-10 22:05:58.422 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-02-10 22:06:02.692 Hail: INFO: Coerced sorted dataset======>(99 + 3) / 100]
2026-02-10 22:06:06.175 Hail: INFO: wrote table with 57902 rows in 100 partitions to gs://gnomad-tmp-4day/rmc/mpc/nonpath_common_gnomad_fitted_scores_grouped_xgb.ht


N rows in common gnomAD grouped fitted score table:
57902


INFO (MPC_experiment_notebook 47): Getting n_less values for input variants...


Number of total gnomAD common variants:
101310


2026-02-10 22:06:31.246 Hail: INFO: wrote table with 70313598 rows in 1000 partitions to gs://gnomad-tmp-4day/rmc/mpc/context_fitted_and_gnomad_scores_xgb.ht
2026-02-10 22:06:39.104 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-02-10 22:07:12.841 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-02-10 22:08:22.188 Hail: INFO: wrote table with 70313598 rows in 1000 partitions to gs://gnomad-tmp-4day/rmc/mpc/context_binary_xgb.ht
INFO (MPC_experiment_notebook 91): Calculating MPC scores...
2026-02-10 22:08:44.530 Hail: INFO: wrote table with 70313598 rows in 1000 partitions to gs://regional_missense_constraint/MPC/4.1/2/mpc_xgb_common_benign_addtl_oe_exp.ht
